# Eka — `founder` persona QLoRA (Colab)

**Generated from `training/train_founder_lora_kaggle.py`** by `scripts/build_colab_notebooks.py`. Edit the script, not this notebook.

**Before you run:**

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Put `secrets.json` at `/content/drive/MyDrive/eka-secrets/secrets.json` with `HF_TOKEN` and `HF_USERNAME`.
3. Run all. The Drive mount will ask you to authorise once.

Output goes to Drive, so a disconnect does not lose checkpoints — re-running resumes from the last one.

In [ ]:
# ---- GPU check ----------------------------------------------
# Colab hands out T4 / L4 / A100 depending on the day. Anything
# Ampere or newer supports bf16; T4 does not, and the script picks
# the compute dtype from this, so it is worth seeing up front.
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU. Runtime -> Change runtime type -> T4 GPU, then '
        'Run all again.'
    )
print(torch.cuda.get_device_name(0))
print('capability', torch.cuda.get_device_capability(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())
print('VRAM GB:', round(
    torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# ---- Drive ---------------------------------------------------
# Credentials come in here, and checkpoints go out here. Kaggle
# used an attached dataset; Colab has no equivalent, and Drive is
# the only store that survives a runtime being recycled.
from google.colab import drive

drive.mount('/content/drive')

import os

os.makedirs('/content/drive/MyDrive/eka_models', exist_ok=True)
secrets_path = '/content/drive/MyDrive/eka-secrets/secrets.json'
assert os.path.exists(secrets_path), (
    f'{secrets_path} not found. Create it with HF_TOKEN and '
    'HF_USERNAME before running.'
)
print('secrets found:', secrets_path)

In [ ]:
# ---- install -------------------------------------------------
# Unpinned and -U on purpose. Colab's preinstalled stack cannot be
# probed from outside, and pinning a stale set against an unknown
# image is exactly what cost the Kaggle runs two sessions. Letting
# pip resolve a consistent modern set is the safer bet; the
# versions print below so a failure has evidence attached.
%pip install -q -U transformers trl peft accelerate datasets bitsandbytes huggingface_hub

import importlib.metadata as _md

for _p in 'transformers trl peft accelerate datasets bitsandbytes huggingface_hub'.split():
    try:
        print(f'{_p:24} {_md.version(_p)}')
    except Exception:
        print(f'{_p:24} (version unavailable)')

import os

# The script installs its own deps unless told not to; this cell
# already did it.
os.environ['EKA_SKIP_INSTALL'] = '1'

In [ ]:
# ---- smoke test toggle ---------------------------------------
# '1' caps training at 12 steps (~15 min) and exits WITHOUT
# pushing an adapter, which is enough to measure the real
# s/step. The persona estimate has already been wrong by 4x
# once; measuring costs a quarter hour, guessing cost twelve.
# Now '0' — the 2026-08-14 smoke test measured 361 s/step and
# the config was tuned off the back of it. Set to '1' to
# re-measure after any change to r / batch / seq len.
import os

os.environ['EKA_SMOKE'] = '0'
print('EKA_SMOKE =', os.environ['EKA_SMOKE'])

## Section 1 — INSTALL

In a notebook, put this in the first cell prefixed with %%capture

In [ ]:
# %%capture
# !pip install -q trl bitsandbytes

import os
import subprocess
import sys


def _pip_install() -> None:
    """Idempotent install so the script works as a plain .py run too."""
    packages = [
        # INSTALL AS LITTLE AS POSSIBLE. Everything else comes from Kaggle's
        # image, which is internally consistent; our pins were not, and every
        # one of them dragged a dependency tree backwards over a stack built
        # for the current versions. Two failures came from exactly that:
        #
        #   numpy<2.0     -> ValueError: numpy.dtype size changed (mixed ABI,
        #                    because the kernel already held 2.x in memory)
        #   bitsandbytes  -> RuntimeError: No module named 'triton.ops'
        #     ==0.43.1       (a mid-2024 build against a Triton 3.6 image)
        #
        # Probed live on 2026-08-13, the image carries transformers 5.0.0,
        # datasets 5.0.0, peft 0.19.1, accelerate 1.13.0, torch 2.10.0+cu128,
        # triton 3.6.0 and numpy 2.0.2. trl and bitsandbytes are the only two
        # things genuinely missing, and unpinned they resolve to trl 1.10.0 /
        # bitsandbytes 0.50.0, which import cleanly against that image.
        #
        # If you pin anything here again, check it against the image first —
        # scripts to probe it are cheap and a wasted GPU session is not.
        "trl",
        "bitsandbytes",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages], check=False
    )


if os.environ.get("EKA_SKIP_INSTALL") != "1":
    _pip_install()

# NumPy 2 is EXPECTED here, not a problem to be fixed. See the note in
# _pip_install above: downgrading it breaks Kaggle's compiled stack. This just
# records which version the session actually got, so a future failure can be
# read against it, and flags that tracking will be off.
import numpy as _np  # noqa: E402

NUMPY_2 = int(_np.__version__.split(".")[0]) >= 2
print(f"✓ numpy {_np.__version__}" + ("  (wandb will be skipped — see below)" if NUMPY_2 else ""))

## Section 2 — AUTH (Kaggle Secrets, with plain-env fallback)

In [ ]:
SECRETS_DATASET = "/content/drive/MyDrive/eka-secrets/secrets.json"


def _load_secrets() -> dict:
    """Read HF_TOKEN / WANDB_API_KEY / HF_USERNAME, dataset first.

    Three sources, in order of how reliably they survive automation:

    1. The attached private dataset (see SECRETS_DATASET). This exists because
       `kaggle kernels push` cannot attach Kaggle Secrets and, worse, DETACHES
       them from a notebook that already had them — so an API-pushed notebook
       always died at this function until someone ticked two boxes in a
       browser. `dataset_sources` IS honoured by push, so this makes an
       unattended launch actually possible.
    2. Kaggle Secrets, for notebooks still driven by hand.
    3. Plain environment, for running the .py outside Kaggle.

    A later source never overwrites an earlier one, so the dataset wins and the
    others stay as fallbacks.
    """
    names = ["HF_TOKEN", "WANDB_API_KEY", "HF_USERNAME"]
    found = {n: "" for n in names}

    # 1. attached dataset
    try:
        import json as _json

        with open(SECRETS_DATASET, "r", encoding="utf-8") as fh:
            blob = _json.load(fh)
        for name in names:
            if not found[name] and blob.get(name):
                found[name] = str(blob[name]).strip()
        print(f"\u2713 secrets from attached dataset ({SECRETS_DATASET})")
    except FileNotFoundError:
        pass
    except Exception as exc:
        print(f"! could not read {SECRETS_DATASET}: {type(exc).__name__}: {exc}")

    # 2. Kaggle Secrets
    if not all(found[n] for n in ("HF_TOKEN", "HF_USERNAME")):
        try:
            raise ImportError("kaggle_secrets is Kaggle-only; Colab reads Drive")

            client = UserSecretsClient()
            for name in names:
                if found[name]:
                    continue
                try:
                    found[name] = client.get_secret(name)
                except Exception:
                    pass
        except Exception:
            pass

    # 3. environment
    for name in names:
        if not found[name]:
            found[name] = os.environ.get(name, "")

    for name, value in found.items():
        if value:
            os.environ[name] = value

    missing = [n for n in ("HF_TOKEN", "HF_USERNAME") if not found.get(n)]
    if missing:
        raise SystemExit(
            f"Missing required secret(s): {', '.join(missing)}\n"
            f"Either attach the eka-secrets dataset (Add Input -> "
            f"aartipanchal01/eka-secrets), or add them under Kaggle -> "
            f"Add-ons -> Secrets, then restart the session."
        )
    return found


SECRETS = _load_secrets()

from huggingface_hub import HfApi, login  # noqa: E402

login(token=SECRETS["HF_TOKEN"])
print("✓ Hugging Face authenticated")

# Experiment tracking is optional and must never be able to end a 3-hour run.
# On 2026-08-13 it did exactly that: wandb reached for np.float_ under NumPy 2
# and the AttributeError propagated straight out of the import, killing founder
# before step 1. The pins above are the fix; this is the seatbelt for the next
# incompatibility, which will not announce itself either. Losing the charts is
# an annoyance; losing the session is 3 hours of a 30 h weekly quota.
USE_WANDB = False
if not SECRETS.get("WANDB_API_KEY"):
    print("! WANDB_API_KEY not set — training without experiment tracking")
elif NUMPY_2:
    # Known-bad rather than discovered-bad: wandb reaches for np.float_, which
    # NumPy 2.0 removed. Skipping the import beats catching its traceback.
    # This is the accepted cost of leaving Kaggle's NumPy 2 alone — the
    # alternative broke the whole session, not just the charts.
    print(
        "! skipping wandb — numpy 2 is live in this session and wandb still "
        "uses np.float_. Training runs, without experiment tracking."
    )
else:
    try:
        import wandb

        wandb.login(key=SECRETS["WANDB_API_KEY"])
        USE_WANDB = True
        print("✓ WandB authenticated")
    except Exception as exc:
        print(
            f"! WandB unavailable ({type(exc).__name__}: {exc}) — "
            f"training without experiment tracking"
        )

## Section 3 — CONFIG

In [ ]:
MODE = "founder"  # <-- the ONLY line to change for the other three personas

HF_USERNAME = os.environ["HF_USERNAME"]
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATASET_REPO = f"{HF_USERNAME}/eka-datasets"
OUTPUT_REPO = f"{HF_USERNAME}/eka-{MODE}-qwen"
OUTPUT_DIR = f"/content/drive/MyDrive/eka_models/{MODE}_lora"

# 1152, not 2048. The longest example in any split is ~1078 tokens, so 2048
# never truncated anything — it just set the padding width. With packing=False
# every batch was padded to the ceiling, so roughly half of all compute went
# into pad tokens.
#
# This is not a micro-optimisation: founder was cancelled at Kaggle's 12h cap
# on 2026-08-14 without finishing ~168 steps, against a 2.5-3.5h estimate.
# 1152 leaves ~70 tokens of headroom over the longest real example.
MAX_SEQ_LEN = 1152
# 8, halved after the 2026-08-14 smoke test measured 361 s/step. Halves
# the trainable parameters (~40M -> ~20M) and the optimiser state with
# them. LORA_ALPHA stays 32, so the scaling factor alpha/r doubles from
# 2 to 4 — that is a real change in effective adapter strength, not a
# free win. Drop alpha to 16 if the persona voice comes out overcooked.
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
# 2, down from 3. A third of the wall clock for the last third of a LoRA's
# convergence is a bad trade when the run does not fit in the session limit at
# all. load_best_model_at_end still picks the better of the two eval points.
EPOCHS = 2
# 4, up from 2. The effective batch is unchanged at 16 (4 x 4), so the
# step count and the learning dynamics stay put — this is purely about
# feeding the GPU better. At micro-batch 2 and seq 1152 a T4 spends much
# of each step underfed; doubling the micro-batch halves the number of
# forward/backward launches per optimiser step.
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch = 16 (4 x 4), same as before
LR = 2e-4
SAVE_STEPS = 50
EVAL_STEPS = 50
WANDB_PROJECT = "eka"
RUN_NAME = f"eka-{MODE}-qwen-v1"

os.makedirs(OUTPUT_DIR, exist_ok=True)
if USE_WANDB:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT

print(f"\n{'=' * 70}")
print(f"  EKA {MODE.upper()} LoRA")
print(f"  base    : {BASE_MODEL}")
print(f"  data    : {DATASET_REPO}  ({MODE}_train.jsonl / {MODE}_val.jsonl)")
print(f"  output  : {OUTPUT_REPO}")
print(f"{'=' * 70}\n")

# Kaggle's "GPU T4 x2" really is two GPUs, and HF Trainer uses ALL visible ones
# by default via nn.DataParallel. The model is loaded with device_map={"": 0},
# pinned to GPU 0, so replica 1 lands on cuda:1 with its weights on cuda:0 and
# the first training step dies with:
#
#     RuntimeError: Caught RuntimeError in replica 1 on device 1.
#     Expected all tensors to be on the same device, but got index is on
#     cuda:1, different from other tensors on cuda:0
#
# Measured 2026-08-13, immediately after TRAINING START.
#
# We want the x2 instance for its RAM ceiling, not for data parallelism — a 7B
# QLoRA fits one T4, and DataParallel across two T4s over PCIe spends more on
# gather/scatter than it gains. Hiding the second GPU is the fix, and it has to
# happen before torch initialises CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch  # noqa: E402

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Kaggle -> Settings -> Accelerator -> GPU T4 x2.\n"
        "(4-bit QLoRA on a 7B model is not viable on CPU.)"
    )

GPU_NAME = torch.cuda.get_device_name(0)

# Fail in five seconds on the wrong GPU rather than ten minutes in, mid-download.
#
# Kaggle's accelerator menu is "GPU T4 x2" or "GPU P100" — there is no T4 x1, so
# asking for a single GPU gets you a P100. That is compute capability 6.0, and
# this image's torch supports 7.0+. Nothing complains at load time: the model
# starts downloading, then bitsandbytes' 4-bit kernel hits a symbol that was
# never compiled for sm_60, prints
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
# and takes the interpreter with it. Papermill reports that as DeadKernelError,
# which reads like an OOM and sends you tuning batch size for no reason.
# Measured 2026-08-13.
CAPABILITY = torch.cuda.get_device_capability(0)
if CAPABILITY < (7, 0):
    raise SystemExit(
        f"{GPU_NAME} is compute capability {CAPABILITY[0]}.{CAPABILITY[1]}; this "
        f"image's torch needs 7.0+.\n"
        "Kaggle -> Settings -> Accelerator -> GPU T4 x2 (NOT GPU P100), then "
        "re-run.\n"
        "The x2 also raises the RAM ceiling, which the 4-bit load wants anyway."
    )
# Turing (T4) has no bfloat16 support. Ampere+ (A100/L4) does. Picking the
# wrong one here is the most common cause of "RuntimeError: expected scalar
# type" or silent NaN losses on Kaggle.
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
print(f"GPU: {GPU_NAME}")
print(f"     bf16 supported: {SUPPORTS_BF16} -> compute dtype {COMPUTE_DTYPE}")
print(f"     VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

## Section 4 — CHECKPOINT RESUME

Kaggle sessions die. This makes a restart cost minutes, not hours.

In [ ]:
import glob  # noqa: E402


def find_latest_checkpoint(directory: str):
    checkpoints = glob.glob(os.path.join(directory, "checkpoint-*"))
    checkpoints = [c for c in checkpoints if os.path.isdir(c)]
    if not checkpoints:
        return None

    def step_of(path: str) -> int:
        try:
            return int(os.path.basename(path).split("-")[-1])
        except ValueError:
            return -1

    return max(checkpoints, key=step_of)


RESUME_FROM = find_latest_checkpoint(OUTPUT_DIR)
if RESUME_FROM:
    print(f"↻ Resuming from {RESUME_FROM}")
else:
    print("→ Starting fresh training")

## Section 5 — LOAD DATA

In [ ]:
from datasets import load_dataset  # noqa: E402

dataset = load_dataset(
    DATASET_REPO,
    data_files={
        "train": f"{MODE}_train.jsonl",
        "validation": f"{MODE}_val.jsonl",
    },
    token=os.environ["HF_TOKEN"],
)
print(f"\nTrain: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

if "text" not in dataset["train"].column_names:
    raise SystemExit(
        f"Expected a 'text' column, got {dataset['train'].column_names}.\n"
        "Re-run ml/scripts/preprocess.py then ml/scripts/upload_to_hf.py."
    )

print("\n--- one training example (truncated) ---")
print(dataset["train"][0]["text"][:600])
print("---------------------------------------\n")

## Section 6 — LOAD MODEL (4-bit NF4 quantization)

In [ ]:
from transformers import (  # noqa: E402
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=os.environ["HF_TOKEN"])

# Do NOT set pad_token = eos_token here. On Llama-3 that was harmless, because
# its eos (<|end_of_text|>) is a different token from the one the template ends
# turns with (<|eot_id|>). On Qwen2.5 they are the same token: eos IS <|im_end|>.
# The SFT collator masks pad positions out of the labels, so pad == eos would
# mask every stop token the model is supposed to be learning, and the adapter
# would never stop generating. Qwen ships a separate <|endoftext|> for padding.
if tokenizer.pad_token is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
    if "<|endoftext|>" in tokenizer.get_vocab():
        tokenizer.pad_token = "<|endoftext|>"
    else:
        print("! no distinct pad token found — stop tokens may be masked in labels")
print(f"     pad={tokenizer.pad_token!r} ({tokenizer.pad_token_id})  "
      f"eos={tokenizer.eos_token!r} ({tokenizer.eos_token_id})")
tokenizer.padding_side = "right"  # left padding corrupts causal LM training

print("Loading base model in 4-bit (first run downloads ~15GB)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},  # pin to GPU 0; "auto" can spill layers to CPU on T4
    trust_remote_code=True,
    token=os.environ["HF_TOKEN"],
)
model.config.use_cache = False  # incompatible with gradient checkpointing
model.config.pretraining_tp = 1
print("✓ Base model loaded")

## Section 7 — LORA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # noqa: E402

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    # All attention + MLP projections. Attention-only (q,v) trains faster but
    # learns persona style noticeably worse.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # expect ~40M trainable / ~0.5% of 7B

## Section 8 — TRAIN

In [ ]:
from trl import SFTConfig, SFTTrainer  # noqa: E402

# SFTConfig replaces TrainingArguments and absorbs what used to be passed to
# SFTTrainer directly. In trl 1.x, dataset_text_field / packing / max_length
# all live on the config, `max_seq_length` no longer exists (it is
# `max_length`), and transformers 5 renamed `evaluation_strategy` to
# `eval_strategy`. Verified against the live Kaggle image, not from memory.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    weight_decay=0.001,
    # Match the quantization compute dtype or loss goes NaN on T4.
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,  # checkpoints land in Drive, which fills faster
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb" if USE_WANDB else "none",
    run_name=RUN_NAME,
    seed=42,
    group_by_length=True,  # big speedup: batches similar-length sequences
    # These four moved here from the SFTTrainer(...) call in trl 1.x.
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,  # persona examples are short; packing blurs turn boundaries
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,  # was tokenizer=, removed in trl 1.x
)

print(f"\n{'=' * 70}")
print(f"  TRAINING START — {RUN_NAME}")
print(f"  ~{len(dataset['train']) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)} optimizer steps")
print(f"{'=' * 70}\n")

# ---------------------------------------------------------------- progress
# Kaggle shows no logs while a kernel runs and none at all if it is cancelled,
# so a slow run is indistinguishable from a healthy one until the session cap
# kills it. This prints a rate and a projection at every logging_steps, and
# mirrors it to Drive, which survives the runtime being recycled.
import time as _time  # noqa: E402

from transformers import TrainerCallback  # noqa: E402

PROGRESS_FILE = "/content/drive/MyDrive/eka_models/progress.txt"


class RateReport(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = _time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.global_step:
            return
        elapsed = _time.time() - self.t0
        per_step = elapsed / state.global_step
        total = state.max_steps or 0
        remaining = (total - state.global_step) * per_step if total else 0
        line = (
            f"step {state.global_step}/{total or '?'}  "
            f"{per_step:.1f}s/step  elapsed {elapsed / 60:.1f}min  "
            f"eta {remaining / 3600:.2f}h  loss={(logs or {}).get('loss', '?')}"
        )
        print("  \u23f1  " + line, flush=True)
        try:
            with open(PROGRESS_FILE, "a", encoding="utf-8") as fh:
                fh.write(line + "\n")
        except Exception:
            pass


trainer.add_callback(RateReport())

# EKA_SMOKE=1 caps the run at a dozen steps: ~15 minutes to learn the real
# s/step before betting a whole session on the estimate. It deliberately does
# NOT push an adapter — a 12-step LoRA is not a model.
if os.environ.get("EKA_SMOKE") == "1":
    trainer.args.max_steps = 12
    print("\n*** SMOKE TEST: capped at 12 steps, no adapter will be pushed ***\n")

trainer.train(resume_from_checkpoint=RESUME_FROM)

SMOKE = os.environ.get("EKA_SMOKE") == "1"
if SMOKE:
    # Deliberately NOT SystemExit. In a notebook that stops only the current
    # cell, so the save cell ran anyway and blew up on a half-torn-down state.
    # Setting a flag lets SECTION 9 decline for a stated reason.
    print("\n*** SMOKE TEST DONE — read the s/step above and multiply. ***")
    print("*** Set EKA_SMOKE = '0' and re-run for a real adapter. ***\n")

metrics = trainer.evaluate()
print(f"\n✓ Final eval loss: {metrics.get('eval_loss'):.4f}")

## Section 9 — SAVE + PUSH TO HUB

In [ ]:
# Two different reasons `trainer` might be unusable here, and they must not be
# treated the same way. A smoke run has nothing worth saving — that is expected
# and we say so. A real run reaching this point without a trainer means
# training itself failed, and the only useful thing to do is stop loudly:
# swallowing it produces an empty repo, a "successful" job, and no traceback,
# which is how three runs cost 20 GPU-hours and explained nothing.
if SMOKE:
    raise SystemExit("Smoke test — no adapter to save. Nothing was pushed.")
if "trainer" not in globals():
    raise SystemExit(
        "trainer is undefined on a REAL run, so training never got that far.\n"
        "The actual error is further up this log. Not saving anything, because "
        "an empty repo that looks like success is worse than a crash."
    )

print("\nSaving adapter...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

card = f"""---
base_model: {BASE_MODEL}
library_name: peft
tags:
- eka
- lora
- qlora
- {MODE}
---

# eka-{MODE}-qwen

QLoRA adapter giving Qwen2.5-7B-Instruct Eka's **{MODE}** persona.

| | |
|---|---|
| base | `{BASE_MODEL}` |
| rank / alpha | {LORA_R} / {LORA_ALPHA} |
| epochs | {EPOCHS} |
| effective batch | {BATCH_SIZE * GRAD_ACCUM} |
| lr / schedule | {LR} cosine |
| max seq len | {MAX_SEQ_LEN} |
| train / val | {len(dataset['train'])} / {len(dataset['validation'])} |
| final eval loss | {metrics.get('eval_loss', float('nan')):.4f} |
| trained on | {GPU_NAME} |

Merge and serve with `ml/scripts/merge_lora.py --mode {MODE}`.
"""
with open(os.path.join(OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as handle:
    handle.write(card)

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=OUTPUT_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=OUTPUT_REPO,
    # Checkpoints are large and already superseded by the final adapter.
    ignore_patterns=["checkpoint-*", "*.pt", "runs/*"],
)
print(f"\n✅ Pushed to HF Hub: https://huggingface.co/{OUTPUT_REPO}")

## Section 10 — SANITY GENERATION

Does it actually sound like the persona? Read the output, don't trust the loss.

In [ ]:
PROBES = {
    "founder": "I have 3 paying customers at 2000/month and 4 months of runway. Should I raise?",
    "chanakya": "My business partner is hiding revenue numbers from me.",
    "gita": "I did everything right and still lost. What was the point?",
    "reflection": "I keep quitting things right before they start working.",
}

model.eval()
prompt = (
    "<|im_start|>user\n"
    f"{PROBES[MODE]}<|im_end|>\n"
    "<|im_start|>assistant\n"
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        top_k=40,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

print(f"\n{'=' * 70}")
print(f"  SANITY CHECK — {MODE}")
print(f"{'=' * 70}")
print(f"USER: {PROBES[MODE]}\n")
print("EKA :", tokenizer.decode(output[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True))
print(f"{'=' * 70}\n")

if USE_WANDB:
    # The adapter is already pushed by this point, so a failure here costs
    # nothing real — but an unhandled one would still mark the whole notebook
    # failed and send you hunting through a successful 3-hour run.
    try:
        wandb.finish()
    except Exception as exc:
        print(f"! wandb.finish() failed ({type(exc).__name__}: {exc}) — ignoring")

print(f"✅ {MODE} DONE. Next: train the remaining personas, then")
print(f"   python ml/scripts/merge_lora.py --mode {MODE}")